# Phase-2:Feature Engineering per LSOA

**Goal:** Build a master feature matrix one row per London LSOA (~4,994), one column per feature.

**Features computed:**
| Feature | Source |
|---|---|
| `crime_count` | Total crimes per LSOA (36 months) |
| `severity_weighted_count` | CCHI-weighted crime count per LSOA |
| `resolution_rate` | % crimes with positive outcome per LSOA |
| `seasonal_volatility` | Std dev of monthly crime count per LSOA |
| `imd_rank` | IMD 2025 overall deprivation rank |
| `income_rank` | IMD 2025 income deprivation rank |
| `employment_rank` | IMD 2025 employment deprivation rank |
| `stop_search_rate` | Stop & searches per km² per LSOA |
| `total_footfall` | Total TfL footfall assigned to each LSOA (36 months) |

**Note on IMD ranks:** Lower rank = more deprived. Direction is inverted in Phase 4 during normalisation.

**Output:** `outputs/phase2/phase2_feature_matrix.parquet`

In [5]:
import pandas as pd
import numpy as np
import geopandas as gpd
import time
from pathlib import Path
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

In [ ]:
# Loading dataset
BASE   = Path('Dataset')
P1     = BASE / 'outputs' / 'phase1'
OUT    = BASE / 'outputs' / 'phase2'
OUT.mkdir(parents=True, exist_ok=True)

SHP_DIR    = BASE / 'LB_shp'
IMD_FILE   = BASE / 'IoD-2025-custom_data_download-LSOA.csv'
GEOCACHE   = OUT / 'station_geocache.csv'

# Cambridge Crime Harm Index weights for police.uk crime categories
# Source: Sherman et al. (2016), adapted for broad police.uk categories
CCHI_WEIGHTS = {
    'Violence and sexual offences': 250,
    'Robbery':                      365,
    'Burglary':                     146,
    'Vehicle crime':                 30,
    'Theft from the person':         10,
    'Shoplifting':                   10,
    'Other theft':                   16,
    'Bicycle theft':                 10,
    'Criminal damage and arson':     91,
    'Drugs':                         16,
    'Public order':                  30,
    'Possession of weapons':        182,
    'Other crime':                    16,
    'Anti-social behaviour':           1,
}

# Positive outcome categories (resolution_rate)
RESOLVED_OUTCOMES = {
    'Suspect charged',
    'Offender given a caution',
    'Offender given a penalty notice',
    'Offender fined',
    'Offender deported',
    'Offender otherwise dealt with',
    'Suspect charged as part of another case',
    'Local resolution',
    'Offender given a drugs possession warning',
    'Offender given conditional discharge',
    'Offender given absolute discharge',
    'Offender sent to prison',
    'Offender given suspended prison sentence',
    'Offender given community sentence',
}

print('Config loaded.')
print('Output folder:', OUT)

Config loaded.
Output folder: C:\Users\mbeck\OneDrive\Documents\CBL-16_data\outputs\phase2


## Section-1: Load Data

In [7]:
print('Loading Phase 1 parquets...')
crimes      = pd.read_parquet(P1 / 'phase1_crimes_london.parquet')
outcomes    = pd.read_parquet(P1 / 'phase1_outcomes_london.parquet')
stop_search = pd.read_parquet(P1 / 'phase1_stop_search_london.parquet')
footfall    = pd.read_parquet(P1 / 'phase1_footfall_monthly.parquet')

print(f'  crimes:      {crimes.shape}')
print(f'  outcomes:    {outcomes.shape}')
print(f'  stop_search: {stop_search.shape}')
print(f'  footfall:    {footfall.shape}')

Loading Phase 1 parquets...
  crimes:      (3443915, 12)
  outcomes:    (2905468, 10)
  stop_search: (372018, 17)
  footfall:    (15615, 3)


In [8]:
print('Loading London LSOA shapefile...')
gdfs   = [gpd.read_file(f) for f in sorted(SHP_DIR.glob('*.shp'))]
london = gpd.GeoDataFrame(pd.concat(gdfs, ignore_index=True), crs=gdfs[0].crs)
london = london[['lsoa21cd', 'lsoa21nm', 'lad22nm', 'geometry']].copy()
london['area_km2'] = london.geometry.area / 1e6  # EPSG:27700 is in metres

LONDON_LSOAS = set(london['lsoa21cd'])
print(f'  Total LSOAs: {len(london)}')
print(f'  Boroughs:    {london["lad22nm"].nunique()}')
print(f'  CRS:         {london.crs}')

Loading London LSOA shapefile...
  Total LSOAs: 4994
  Boroughs:    33
  CRS:         EPSG:27700


In [9]:
print('Loading IMD 2025...')
imd_raw = pd.read_csv(IMD_FILE)
imd = imd_raw[[
    'LSOA code (2021)',
    'Index of Multiple Deprivation (IMD) Rank',
    'Income Rank',
    'Employment Rank',
]].copy()
imd.columns = ['lsoa21cd', 'imd_rank', 'income_rank', 'employment_rank']
print(f'  IMD shape: {imd.shape}')
print(f'  Nulls: {imd.isnull().sum().to_dict()}')

Loading IMD 2025...
  IMD shape: (33755, 4)
  Nulls: {'lsoa21cd': 0, 'imd_rank': 0, 'income_rank': 0, 'employment_rank': 0}


## Section-2: Filter Crimes to London LSOAs
The crime data contains records outside London (Met Police covers parts of Surrey/Hertfordshire). Filter to LSOAs present in the shapefile.

In [10]:
before = len(crimes)
crimes_london = crimes[crimes['LSOA code'].isin(LONDON_LSOAS)].copy()
crimes_london = crimes_london.rename(columns={'LSOA code': 'lsoa21cd'})
print(f'Crimes before filter: {before:,}')
print(f'Crimes after filter:  {len(crimes_london):,}  ({before - len(crimes_london):,} outside London dropped)')
print(f'Unique London LSOAs in crime data: {crimes_london["lsoa21cd"].nunique()}')

Crimes before filter: 3,443,915
Crimes after filter:  3,400,717  (43,198 outside London dropped)
Unique London LSOAs in crime data: 4981


## Section-3: Crime Count & Severity-Weighted Count

In [11]:
# Apply CCHI weights
crimes_london['cchi_weight'] = crimes_london['Crime type'].map(CCHI_WEIGHTS).fillna(16)  # default=Other

crime_features = (
    crimes_london
    .groupby('lsoa21cd')
    .agg(
        crime_count=('Crime ID', 'count'),
        severity_weighted_count=('cchi_weight', 'sum'),
    )
    .reset_index()
)

print(f'Shape: {crime_features.shape}')
print(crime_features.describe().round(1))
crime_features.head(3)

Shape: (4981, 3)
       crime_count  severity_weighted_count
count       4981.0                   4981.0
mean         543.7                  60153.3
std         1108.2                  73744.5
min           20.0                   2407.0
25%          233.0                  29113.0
50%          357.0                  45475.0
75%          570.0                  68847.0
max        40511.0                1891536.0


,lsoa21cd,crime_count,severity_weighted_count
0,E01000001,519,30728
1,E01000002,1391,73342
2,E01000003,444,44886


## Section-4: Resolution Rate per LSOA

In [ ]:
# Join outcomes to crimes on Crime ID
# Keep only crimes with a non-null Crime ID (ASB records don't have one)

crimes_with_id = crimes_london[crimes_london['Crime ID'].notna()][['Crime ID', 'lsoa21cd']].copy()
outcomes_clean = outcomes[outcomes['Crime ID'].notna()][['Crime ID', 'Outcome type']].copy()

# Left join — crimes without an outcome record stay in (unresolved/pending)
merged = crimes_with_id.merge(outcomes_clean, on='Crime ID', how='left')
merged['resolved'] = merged['Outcome type'].apply(
    lambda x: any(r.lower() in str(x).lower() for r in RESOLVED_OUTCOMES) if pd.notna(x) else False
)

resolution = (
    merged
    .groupby('lsoa21cd')
    .agg(
        crimes_with_outcome=('Crime ID', 'count'),
        resolved_count=('resolved', 'sum'),
    )
    .reset_index()
)
resolution['resolution_rate'] = (
    resolution['resolved_count'] / resolution['crimes_with_outcome'] * 100
).round(2)

resolution = resolution[['lsoa21cd', 'resolution_rate']]
print(f'Shape: {resolution.shape}')
print(resolution['resolution_rate'].describe().round(2))
resolution.head(3)

Shape: (4981, 2)
count    4981.00
mean        7.90
std         3.72
min         0.00
25%         5.50
50%         7.47
75%         9.69
max        47.57
Name: resolution_rate, dtype: float64


,lsoa21cd,resolution_rate
0,E01000001,11.15
1,E01000002,14.65
2,E01000003,7.49


## Section-5: Seasonal Volatility
Standard devation (Std dev) of monthly crime count per LSOA across 36 months.

In [13]:
# Monthly crime count per LSOA
monthly_counts = (
    crimes_london
    .groupby(['lsoa21cd', 'Month'])
    .size()
    .reset_index(name='monthly_crime_count')
)

# Fill missing months with 0 for LSOAs that had no crime in that month
all_months  = crimes_london['Month'].sort_values().unique()
all_lsoas   = crimes_london['lsoa21cd'].unique()
full_index  = pd.MultiIndex.from_product([all_lsoas, all_months], names=['lsoa21cd', 'Month'])
monthly_counts = (
    monthly_counts
    .set_index(['lsoa21cd', 'Month'])
    .reindex(full_index, fill_value=0)
    .reset_index()
)

volatility = (
    monthly_counts
    .groupby('lsoa21cd')['monthly_crime_count']
    .std()
    .reset_index(name='seasonal_volatility')
)

print(f'Shape: {volatility.shape}')
print(volatility['seasonal_volatility'].describe().round(2))
volatility.head(3)

Shape: (4981, 2)
count    4981.00
mean        5.87
std         7.88
min         1.12
25%         3.44
50%         4.56
75%         6.43
max       364.09
Name: seasonal_volatility, dtype: float64


,lsoa21cd,seasonal_volatility
0,E01000001,4.704861
1,E01000002,8.449805
2,E01000003,6.257694


## Section-6: Stop & Search Rate (spatial join → per km²)

In [14]:
# Filter stop_search to project period Apr 2023 – Mar 2026
ss = stop_search[
    (stop_search['Month'] >= '2023-04') &
    (stop_search['Month'] <= '2026-03')
].copy()

# Create GeoDataFrame — stop_search coords are WGS84 (EPSG:4326)
ss_gdf = gpd.GeoDataFrame(
    ss,
    geometry=gpd.points_from_xy(ss['Longitude'], ss['Latitude']),
    crs='EPSG:4326'
)
# Reproject to match London shapefile (EPSG:27700)
ss_gdf = ss_gdf.to_crs('EPSG:27700')

# Spatial join: assign each stop to an LSOA
ss_joined = gpd.sjoin(
    ss_gdf[['geometry']],
    london[['lsoa21cd', 'area_km2', 'geometry']],
    how='left',
    predicate='within'
)

# Count per LSOA and normalise by area
ss_counts = (
    ss_joined
    .dropna(subset=['lsoa21cd'])
    .groupby('lsoa21cd')
    .agg(ss_total=('lsoa21cd', 'count'), area_km2=('area_km2', 'first'))
    .reset_index()
)
ss_counts['stop_search_rate'] = (ss_counts['ss_total'] / ss_counts['area_km2']).round(4)
ss_counts = ss_counts[['lsoa21cd', 'stop_search_rate']]

matched_pct = len(ss_joined.dropna(subset=['lsoa21cd'])) / len(ss_gdf) * 100
print(f'Stop & searches matched to an LSOA: {matched_pct:.1f}%')
print(f'Shape: {ss_counts.shape}')
print(ss_counts['stop_search_rate'].describe().round(3))
ss_counts.head(3)

Stop & searches matched to an LSOA: 99.7%
Shape: (4969, 2)
count     4969.000
mean       443.100
std        836.202
min          0.348
25%         63.276
50%        173.132
75%        471.892
max      15789.837
Name: stop_search_rate, dtype: float64


,lsoa21cd,stop_search_rate
0,E01000001,1031.8383
1,E01000002,752.9996
2,E01000003,762.0118


## Section-7: Footfall (geocode stations → spatial join → sum per LSOA)

TfL station coordinates are not in the footfall CSV. We geocode station names using Nominatim (OpenStreetMap), then spatially join to LSOAs. Results are cached to avoid re-geocoding.

In [15]:
stations = footfall['Station'].unique()
print(f'Unique stations to geocode: {len(stations)}')

# Load cache if it exists
if GEOCACHE.exists():
    cache_df = pd.read_csv(GEOCACHE)
    geocoded = dict(zip(cache_df['station'], zip(cache_df['lat'], cache_df['lon'])))
    print(f'Loaded {len(geocoded)} cached stations')
else:
    geocoded = {}
    print('No cache found — will geocode all stations')

to_geocode = [s for s in stations if s not in geocoded]
print(f'Stations still to geocode: {len(to_geocode)}')

Unique stations to geocode: 437
No cache found — will geocode all stations
Stations still to geocode: 437


In [16]:
if to_geocode:
    geolocator = Nominatim(user_agent='cbl16_project')
    geocode    = RateLimiter(geolocator.geocode, min_delay_seconds=1.1)

    print(f'Geocoding {len(to_geocode)} stations (this will take ~{len(to_geocode)//60 + 1} mins)...')
    failed = []
    for i, station in enumerate(to_geocode):
        query = f'{station} station, London, UK'
        try:
            loc = geocode(query)
            if loc:
                geocoded[station] = (loc.latitude, loc.longitude)
            else:
                # Retry with simplified query
                loc2 = geocode(f'{station}, London')
                if loc2:
                    geocoded[station] = (loc2.latitude, loc2.longitude)
                else:
                    failed.append(station)
        except Exception as e:
            failed.append(station)

        if (i + 1) % 50 == 0:
            print(f'  {i+1}/{len(to_geocode)} done...')

    # Save cache
    cache_rows = [{'station': s, 'lat': v[0], 'lon': v[1]} for s, v in geocoded.items()]
    pd.DataFrame(cache_rows).to_csv(GEOCACHE, index=False)
    print(f'\nGeocoded: {len(geocoded)} | Failed: {len(failed)}')
    if failed:
        print('Failed stations:', failed)
else:
    print('All stations already cached.')

Geocoding 437 stations (this will take ~8 mins)...


RateLimiter caught an error, retrying (0/2 tries). Called with (*('Baker Street station, London, UK',), **{}).
Traceback (most recent call last):
  File "c:\Users\mbeck\AppData\Local\Programs\Python\Python310\lib\site-packages\urllib3\connectionpool.py", line 449, in _make_request
    six.raise_from(e, None)
  File "<string>", line 3, in raise_from
  File "c:\Users\mbeck\AppData\Local\Programs\Python\Python310\lib\site-packages\urllib3\connectionpool.py", line 444, in _make_request
    httplib_response = conn.getresponse()
  File "c:\Users\mbeck\AppData\Local\Programs\Python\Python310\lib\http\client.py", line 1368, in getresponse
    response.begin()
  File "c:\Users\mbeck\AppData\Local\Programs\Python\Python310\lib\http\client.py", line 317, in begin
    version, status, reason = self._read_status()
  File "c:\Users\mbeck\AppData\Local\Programs\Python\Python310\lib\http\client.py", line 278, in _read_status
    line = str(self.fp.readline(_MAXLINE + 1), "iso-8859-1")
  File "c:\Users

  50/437 done...


RateLimiter caught an error, retrying (0/2 tries). Called with (*('Canada Water station, London, UK',), **{}).
Traceback (most recent call last):
  File "c:\Users\mbeck\AppData\Local\Programs\Python\Python310\lib\site-packages\urllib3\connectionpool.py", line 449, in _make_request
    six.raise_from(e, None)
  File "<string>", line 3, in raise_from
  File "c:\Users\mbeck\AppData\Local\Programs\Python\Python310\lib\site-packages\urllib3\connectionpool.py", line 444, in _make_request
    httplib_response = conn.getresponse()
  File "c:\Users\mbeck\AppData\Local\Programs\Python\Python310\lib\http\client.py", line 1368, in getresponse
    response.begin()
  File "c:\Users\mbeck\AppData\Local\Programs\Python\Python310\lib\http\client.py", line 317, in begin
    version, status, reason = self._read_status()
  File "c:\Users\mbeck\AppData\Local\Programs\Python\Python310\lib\http\client.py", line 278, in _read_status
    line = str(self.fp.readline(_MAXLINE + 1), "iso-8859-1")
  File "c:\Users

  100/437 done...


...

  350/437 done...


RateLimiter caught an error, retrying (0/2 tries). Called with (*('Sudbury Town station, London, UK',), **{}).
Traceback (most recent call last):
  File "c:\Users\mbeck\AppData\Local\Programs\Python\Python310\lib\site-packages\urllib3\connectionpool.py", line 449, in _make_request
    six.raise_from(e, None)
  File "<string>", line 3, in raise_from
  File "c:\Users\mbeck\AppData\Local\Programs\Python\Python310\lib\site-packages\urllib3\connectionpool.py", line 444, in _make_request
    httplib_response = conn.getresponse()
  File "c:\Users\mbeck\AppData\Local\Programs\Python\Python310\lib\http\client.py", line 1368, in getresponse
    response.begin()
  File "c:\Users\mbeck\AppData\Local\Programs\Python\Python310\lib\http\client.py", line 317, in begin
    version, status, reason = self._read_status()
  File "c:\Users\mbeck\AppData\Local\Programs\Python\Python310\lib\http\client.py", line 278, in _read_status
    line = str(self.fp.readline(_MAXLINE + 1), "iso-8859-1")
  File "c:\Users

  400/437 done...


RateLimiter caught an error, retrying (0/2 tries). Called with (*('West Hampstead LO, London',), **{}).
Traceback (most recent call last):
  File "c:\Users\mbeck\AppData\Local\Programs\Python\Python310\lib\site-packages\urllib3\connectionpool.py", line 449, in _make_request
    six.raise_from(e, None)
  File "<string>", line 3, in raise_from
  File "c:\Users\mbeck\AppData\Local\Programs\Python\Python310\lib\site-packages\urllib3\connectionpool.py", line 444, in _make_request
    httplib_response = conn.getresponse()
  File "c:\Users\mbeck\AppData\Local\Programs\Python\Python310\lib\http\client.py", line 1368, in getresponse
    response.begin()
  File "c:\Users\mbeck\AppData\Local\Programs\Python\Python310\lib\http\client.py", line 317, in begin
    version, status, reason = self._read_status()
  File "c:\Users\mbeck\AppData\Local\Programs\Python\Python310\lib\http\client.py", line 278, in _read_status
    line = str(self.fp.readline(_MAXLINE + 1), "iso-8859-1")
  File "c:\Users\mbeck\


Geocoded: 411 | Failed: 26
Failed stations: ['Burnham Bucks', 'Canary Wharf EL', 'Carpenders Park', 'Chalfont & Latimer', 'Custom House DLR', 'Custom House EL', 'Edgware Road C&H', 'Hammersmith C&H', 'Hammersmith D&P', 'Heathrow T2&3 TfL Rail/HEx', 'Heathrow T4 TfL Rail/HEx', 'Heathrow T5 TfL Rail/HEx', 'Langley Berks', 'Paddington EL', 'Pontoon Dock DLR', 'Shadwell DLR', 'Shepherds Bush Market', 'Sydenham SR', 'Theobalds Grove', 'Theydon Bois', 'Watford High Street', 'Watford Met', 'Woolwich EL', 'Canary Wharf Elizabeth Line', 'Custom House Elizabeth Line', 'Woolwich Elizabeth Line']


In [23]:
# Build station GeoDataFrame
station_rows = [
    {'station': s, 'lat': v[0], 'lon': v[1]}
    for s, v in geocoded.items()
]
station_gdf = gpd.GeoDataFrame(
    pd.DataFrame(station_rows),
    geometry=gpd.points_from_xy(
        [r['lon'] for r in station_rows],
        [r['lat'] for r in station_rows]
    ),
    crs='EPSG:4326'
).to_crs('EPSG:27700')

# Spatial join: assign each station to an LSOA
station_lsoa = gpd.sjoin(
    station_gdf[['station', 'geometry']],
    london[['lsoa21cd', 'geometry']],
    how='left',
    predicate='within'
)[['station', 'lsoa21cd']]

matched_stations = station_lsoa['lsoa21cd'].notna().sum()
print(f'Stations matched to an LSOA: {matched_stations} / {len(station_gdf)}')

# Join LSOA back to footfall and sum per LSOA across all months
footfall_lsoa = (
    footfall
    .merge(station_lsoa, left_on='Station', right_on='station', how='left')
    .dropna(subset=['lsoa21cd'])
    .groupby('lsoa21cd')['TotalFootfall']
    .sum()
    .reset_index(name='total_footfall')
)

print(f'LSOAs with footfall data: {len(footfall_lsoa)}')
print(footfall_lsoa['total_footfall'].describe().round(0))
footfall_lsoa.head(3)

Stations matched to an LSOA: 401 / 411
LSOAs with footfall data: 371
count          371.0
mean      21480783.0
std       38627441.0
min         606214.0
25%        5360048.0
50%       10497400.0
75%       19595488.0
max      418933989.0
Name: total_footfall, dtype: float64


,lsoa21cd,total_footfall
0,E01000001,15567714
1,E01000002,74117132
2,E01000005,27129277


## Section-8: Assemble Master Feature Matrix

In [24]:
# Start from master LSOA list (shapefile)
feature_matrix = london[['lsoa21cd', 'lsoa21nm', 'lad22nm', 'area_km2']].copy()

# Join all features
feature_matrix = feature_matrix.merge(crime_features,  on='lsoa21cd', how='left')
feature_matrix = feature_matrix.merge(resolution,      on='lsoa21cd', how='left')
feature_matrix = feature_matrix.merge(volatility,      on='lsoa21cd', how='left')
feature_matrix = feature_matrix.merge(imd,             on='lsoa21cd', how='left')
feature_matrix = feature_matrix.merge(ss_counts,       on='lsoa21cd', how='left')
feature_matrix = feature_matrix.merge(footfall_lsoa,   on='lsoa21cd', how='left')

# Fill nulls with 0 for count-based features (LSOAs with no events)
fill_zero = ['crime_count', 'severity_weighted_count', 'seasonal_volatility',
             'stop_search_rate', 'total_footfall']
feature_matrix[fill_zero] = feature_matrix[fill_zero].fillna(0)

# resolution_rate: LSOAs with no crimes get NaN (can't compute rate)
# We'll leave as NaN and handle in Phase 3

print(f'Shape: {feature_matrix.shape}')
print(f'\nNull counts:')
print(feature_matrix.isnull().sum())
feature_matrix.head(3)

Shape: (4994, 13)

Null counts:
lsoa21cd                    0
lsoa21nm                    0
lad22nm                     0
area_km2                    0
crime_count                 0
severity_weighted_count     0
resolution_rate            13
seasonal_volatility         0
imd_rank                    0
income_rank                 0
employment_rank             0
stop_search_rate            0
total_footfall              0
dtype: int64


,lsoa21cd,lsoa21nm,lad22nm,area_km2,crime_count,severity_weighted_count,resolution_rate,seasonal_volatility,imd_rank,income_rank,employment_rank,stop_search_rate,total_footfall
0,E01000011,Barking and Dagenham 016C,Barking and Dagenham,0.091632,219.0,38015.0,7.82,2.768731,5535,4637,11485,1615.1513,0.0
1,E01000046,Barking and Dagenham 017D,Barking and Dagenham,0.155494,323.0,51544.0,14.36,5.140873,9497,5960,9319,128.6223,0.0
2,E01000051,Barking and Dagenham 021D,Barking and Dagenham,0.066632,331.0,50527.0,6.47,4.125030,7571,1649,17031,1140.5846,0.0


In [25]:
# Descriptive summary
numeric_cols = ['crime_count', 'severity_weighted_count', 'resolution_rate',
                'seasonal_volatility', 'imd_rank', 'income_rank',
                'employment_rank', 'stop_search_rate', 'total_footfall']
print(feature_matrix[numeric_cols].describe().round(2).to_string())

       crime_count  severity_weighted_count  resolution_rate  seasonal_volatility  imd_rank  income_rank  employment_rank  stop_search_rate  total_footfall
count      4994.00                  4994.00          4981.00              4994.00   4994.00      4994.00          4994.00           4994.00    4.994000e+03
mean        542.24                 59996.76             7.90                 5.85  15349.43     13675.39         16597.41            440.88    1.595789e+06
std        1107.13                 73712.17             3.72                 7.87   8631.59      9019.70          8733.24            834.69    1.192926e+07
min           0.00                     0.00             0.00                 0.00    238.00         1.00             1.00              0.00    0.000000e+00
25%         232.00                 29005.50             5.50                 3.44   8110.50      6346.00          9462.75             62.34    0.000000e+00
50%         356.00                 45383.50             7.47    

## Section-9: Save Outputs

In [26]:
# Save feature matrix (without geometry — pure tabular)
feature_matrix.to_parquet(OUT / 'phase2_feature_matrix.parquet', index=False)
print(f'Saved phase2_feature_matrix.parquet — {feature_matrix.shape}')

# Also save monthly crime counts per LSOA (used in Phase 3 for STL/Kruskal)
monthly_counts.to_parquet(OUT / 'phase2_monthly_crime_counts.parquet', index=False)
print(f'Saved phase2_monthly_crime_counts.parquet — {monthly_counts.shape}')

print('\nPhase 2 complete.')

Saved phase2_feature_matrix.parquet — (4994, 13)
Saved phase2_monthly_crime_counts.parquet — (179316, 3)

Phase 2 complete.


In [27]:
# --- Phase 2 summary ---
print('=' * 55)
print('PHASE 2 SUMMARY')
print('=' * 55)
print(f'Total LSOAs in matrix:          {len(feature_matrix):>6}')
print(f'LSOAs with crime data:          {(feature_matrix["crime_count"] > 0).sum():>6}')
print(f'LSOAs with stop & search data:  {(feature_matrix["stop_search_rate"] > 0).sum():>6}')
print(f'LSOAs with footfall data:       {(feature_matrix["total_footfall"] > 0).sum():>6}')
print(f'LSOAs with IMD data:            {feature_matrix["imd_rank"].notna().sum():>6}')
print(f'LSOAs with resolution rate:     {feature_matrix["resolution_rate"].notna().sum():>6}')
print('=' * 55)
print(f'Outputs saved to: {OUT}')

PHASE 2 SUMMARY
Total LSOAs in matrix:            4994
LSOAs with crime data:            4981
LSOAs with stop & search data:    4969
LSOAs with footfall data:          371
LSOAs with IMD data:              4994
LSOAs with resolution rate:       4981
Outputs saved to: C:\Users\mbeck\OneDrive\Documents\CBL-16_data\outputs\phase2
